# 1.1.8 File I/O

CSV, JSON, Pickle, pathlib — applied to the Titanic dataset.

In [ ]:
import csv, json, pickle, urllib.request
from pathlib import Path

DATA = Path('data'); DATA.mkdir(exist_ok=True)
dest = DATA / 'titanic.csv'
if not dest.exists():
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/phihung/titanic/resolve/main/train.csv', dest)
print('File size:', dest.stat().st_size, 'bytes')

In [ ]:
# CSV read + clean
with open(dest, newline='') as f:
    rows = list(csv.DictReader(f))
print('Columns:', list(rows[0].keys()))
print('Row count:', len(rows))

cleaned = []
for r in rows:
    try: cleaned.append({'id': int(r['PassengerId']), 'survived': int(r['Survived']),
                          'class': int(r['Pclass']), 'age': float(r['Age'] or 0), 'fare': float(r['Fare'] or 0)})
    except: pass
print('Cleaned rows:', len(cleaned))

In [ ]:
# JSON artifact
import hashlib; from datetime import datetime, timezone
artifact = {'rows': len(cleaned), 'survived': sum(r['survived'] for r in cleaned),
            'avg_fare': round(sum(r['fare'] for r in cleaned)/len(cleaned), 2),
            'checksum': hashlib.sha256(str(cleaned[:10]).encode()).hexdigest()[:16],
            'ts': datetime.now(timezone.utc).isoformat()}
out = Path('output'); out.mkdir(exist_ok=True)
(out / 'artifact.json').write_text(json.dumps(artifact, indent=2))
print(json.dumps(artifact, indent=2))

In [ ]:
# Pickle round-trip
pkl = out / 'rows.pkl'
with open(pkl, 'wb') as f: pickle.dump(cleaned, f)
with open(pkl, 'rb') as f: loaded = pickle.load(f)
print(f'Pickled {len(loaded)} rows → {pkl.stat().st_size/1024:.1f} KB')

In [ ]:
# pathlib operations
print('output/ contents:')
for f in sorted(out.iterdir()):
    print(f'  {f.name:<30} {f.stat().st_size:>8} bytes')